# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR² dataset package using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described by a Croissant schema available at the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata
metadata = dataset.metadata

# Display core dataset description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Explore available record sets in the dataset, and inspect their fields along with their `@id` values.

**Note:** In this notebook, all entities (record sets, fields, columns) are referenced by their unique `@id` as defined in the Croissant schema.

In [ ]:
# Discover available record set @ids
record_set_ids = [rs['@id'] for rs in metadata.to_json().get('recordSet', [])]
if not record_set_ids:
    # Try new property name if present
    record_set_ids = [rs['@id'] for rs in metadata.to_json().get('recordSets', [])]

if not record_set_ids:
    print('No record sets found in schema.')
else:
    print('Available record set @ids:')
    for rid in record_set_ids:
        print(f' - {rid}')

    # For each record set, show fields and columns by @id
    for rid in record_set_ids:
        print(f'\nInspecting record set @id: {rid}')
        # Find record set metadata
        record_set = next((rs for rs in metadata.to_json().get('recordSet', []) if rs['@id']==rid), None)
        if record_set is None:
            print('  (record set metadata unavailable)')
            continue
        fields = [f['@id'] for f in record_set.get('field', [])]
        print(f'  Fields: {fields}')
        columns = []
        # Try to discover columns for tabular record sets
        if 'column' in record_set:
            columns = [c['@id'] for c in record_set['column']]
        if columns:
            print(f'  Columns: {columns}')

## 3. Data Extraction
Load data from each record set into a DataFrame for subsequent analysis.

In all steps below, use the `@id` value (as shown above) for record sets and columns/fields.

In [ ]:
# Load records for each available record set into a pandas DataFrame
dataframes = dict()
if not record_set_ids:
    print('No record sets to load records from.')
else:
    for record_set_id in record_set_ids:
        print(f'Loading records from record set: {record_set_id}')
        try:
            records = list(dataset.records(record_set=record_set_id))
        except Exception as ex:
            print(f'  Could not load records: {ex}')
            continue
        if records:
            df = pd.DataFrame(records)
            print(f'  Records loaded: {len(df)}')
            print(f'  Columns: {df.columns.tolist()}')
            dataframes[record_set_id] = df
            print(df.head(3))
        else:
            print('  No records found.')

# Choose the primary record set for further analysis
primary_record_set_id = None
if dataframes:
    # Take the first loaded record set as primary
    primary_record_set_id = list(dataframes.keys())[0]
    print(f'Using {primary_record_set_id} for subsequent steps.')
else:
    print('No tabular record sets found for later steps.')

## 4. Exploratory Data Analysis (EDA)
Apply common analysis and preprocessing steps: filter on a numeric field, remove outliers, normalize values, and group by categorical fields using `@id` reference.

In [ ]:
# EDA: Filtering, normalization, and grouping
if not primary_record_set_id or primary_record_set_id not in dataframes:
    print('No data available for EDA.')
else:
    df = dataframes[primary_record_set_id]
    # Display columns for manual inspection
    print('Available columns:', df.columns.tolist())
    # Attempt to select a sensible numeric field
    possible_numeric_fields = [col for col in df.columns if df[col].dtype.kind in 'iufc']
    if not possible_numeric_fields:
        print('No numeric fields found for EDA.')
    else:
        numeric_field_id = possible_numeric_fields[0]
        print(f'Using numeric field for EDA: {numeric_field_id}')

        # Filter: Example (values > mean, as placeholder since domain min may vary)
        if df[numeric_field_id].dtype.kind not in 'fiu':
            df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f'Filtered records where {numeric_field_id} > {threshold:.2f}: {len(filtered_df)} records')
        print(filtered_df.head(3))

        # Normalization
        norm_col = f'{numeric_field_id}_normalized'
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f'Normalized {numeric_field_id} for filtered records:')
        print(filtered_df[[numeric_field_id, norm_col]].head(3))

        # Group by a categorical field, if available
        possible_categorical_fields = [col for col in df.columns if df[col].dtype.name == 'object' and col != numeric_field_id]
        if possible_categorical_fields:
            group_field_id = possible_categorical_fields[0]
            print(f'Grouping by {group_field_id} (example):')
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(grouped_df.head())
        else:
            print('No categorical fields available for grouping.')

## 5. Visualization
Visualize the distribution of a numeric field and its relation to a categorical variable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not primary_record_set_id or primary_record_set_id not in dataframes:
    print('No data available for visualization.')
else:
    df = dataframes[primary_record_set_id]
    numeric_fields = [col for col in df.columns if df[col].dtype.kind in 'iufc']
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        # Histogram
        plt.figure(figsize=(7,4))
        sns.histplot(df[numeric_field_id].dropna(), kde=True)
        plt.title(f'Distribution of {numeric_field_id}')
        plt.xlabel(numeric_field_id)
        plt.show()

        # Boxplot by a group field if available
        categorical_fields = [col for col in df.columns if df[col].dtype.name == 'object' and col != numeric_field_id]
        if categorical_fields:
            group_field = categorical_fields[0]
            plt.figure(figsize=(8,4))
            sns.boxplot(x=df[group_field], y=df[numeric_field_id])
            plt.title(f'{numeric_field_id} by {group_field}')
            plt.xlabel(group_field)
            plt.ylabel(numeric_field_id)
            plt.xticks(rotation=45, ha='right')
            plt.tight_layout()
            plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load a FAIR² Croissant dataset using `mlcroissant`, review its structure by `@id`, and extract data for initial exploratory analysis and visualization. You can continue this workflow to conduct domain-specific statistical evaluations and build predictive models, leveraging the unambiguous referencing by record set and field/column `@id` throughout.